# Walsh-Hadamard Diagonal Unitary Synthesis

A unitary diagonal operator $D = \mathrm{diag}(e^{i\theta_0}, \ldots, e^{i\theta_{2^n-1}})$ can be written as a product of commuting $Z$-parity phase gadgets:
$$D = \prod_{k=0}^{2^n-1} e^{i a_k W_k}$$
where $W_k = \bigotimes_{j:\,k_j=1} Z_j$ acts on the qubits selected by the set bits of $k$, and the coefficients $a_k$ are given by the **Fast Walsh-Hadamard Transform** (FWHT) of the phase vector $(\theta_j)$:
$$a_k = \frac{1}{2^n} \sum_{j=0}^{2^n-1} (-1)^{\mathrm{popcount}(j \mathbin{\&} k)}\, \theta_j$$

Each non-global term $e^{i a_k W_k}$ is realized as a **parity phase gadget**: a CNOT ladder that XORs the support qubits onto a target, an $R_z$ rotation on the target, and the inverse ladder.

In [ ]:
import numpy as np
from guppyalgos.algorithms.state_preparation.diagonal import diagonal_unitary_walsh, fast_walsh_hadamard_transform
from guppyalgos.tests.helpers import Endianness, assert_allclose_ignorephase, get_unitary

## 1-qubit example: S gate

The S gate is $\mathrm{diag}(1, i)$. Its FWHT gives a single non-global term, implemented as one $R_z$ rotation.

In [2]:
diagonal_1q = np.array([1.0, 1j])
phases_1q = np.angle(diagonal_1q)
walsh_coeffs_1q = fast_walsh_hadamard_transform(phases_1q)
print("Phases:         ", phases_1q)
print("Walsh coeffs:   ", walsh_coeffs_1q)
print("Normalized a_k: ", walsh_coeffs_1q / len(diagonal_1q))

Phases:          [0.         1.57079633]
Walsh coeffs:    [ 1.57079633 -1.57079633]
Normalized a_k:  [ 0.78539816 -0.78539816]


In [ ]:
circ_1q = diagonal_unitary_walsh(diagonal_1q)
unitary_1q = get_unitary(circ_1q, 1, endianness=Endianness.LITTLE)
print("Synthesized unitary:\n", unitary_1q)
assert_allclose_ignorephase(np.diag(diagonal_1q), unitary_1q)
print("Matches S gate up to global phase. ✓")

## 2-qubit example: CZ gate

The CZ gate is $\mathrm{diag}(1, 1, 1, -1)$. Its FWHT produces three non-global terms, each implemented as a parity gadget.

In [4]:
diagonal_2q = np.array([1.0, 1.0, 1.0, -1.0])
phases_2q = np.angle(diagonal_2q)
walsh_coeffs_2q = fast_walsh_hadamard_transform(phases_2q)
n_terms = np.sum(np.abs(walsh_coeffs_2q / len(diagonal_2q)) > 1e-10)
print("Walsh coeffs: ", walsh_coeffs_2q)
print(f"Non-trivial terms (excluding global phase): {n_terms - 1}")

Walsh coeffs:  [ 3.14159265 -3.14159265 -3.14159265  3.14159265]
Non-trivial terms (excluding global phase): 3


In [ ]:
circ_2q = diagonal_unitary_walsh(diagonal_2q)
unitary_2q = get_unitary(circ_2q, 2, endianness=Endianness.LITTLE)
print("Synthesized unitary:\n", np.round(unitary_2q, 6))
assert_allclose_ignorephase(np.diag(diagonal_2q), unitary_2q)
print("Matches CZ gate up to global phase. ✓")

## Approximate synthesis via truncation

Setting a higher `truncation_threshold` drops small Walsh terms, yielding a shallower approximate circuit. The quality of the approximation depends on how concentrated the Walsh spectrum is.

In [ ]:
rng = np.random.default_rng(42)
diagonal_generic = np.exp(1j * rng.uniform(-np.pi, np.pi, 4))

circ_exact = diagonal_unitary_walsh(diagonal_generic, truncation_threshold=0.0)
circ_approx = diagonal_unitary_walsh(diagonal_generic, truncation_threshold=0.3)

u_exact = get_unitary(circ_exact, 2, endianness=Endianness.LITTLE)
u_approx = get_unitary(circ_approx, 2, endianness=Endianness.LITTLE)

error = np.linalg.norm(u_exact - u_approx)
print(f"Frobenius error from truncation: {error:.4f}")